# Rubin/LSST — Stable Stars in DDFs: Image Cutouts from CDS services

For each photometrically stable star with known spectral type (from the
Simbad master catalogue built in notebook `01_findStarsinSimbad.ipynb`
and cross-matched with Rubin visits in `02_StableStars_inRubinVisits.ipynb`),
this notebook retrieves a sky image cutout of configurable size (~10 arcsec)
centred on the star's position using the **CDS HiPS2FITS** service
(Centre de Données astronomiques de Strasbourg, Strasbourg Astronomical Data Center).

The HiPS2FITS web service (`https://alasky.cds.unistra.fr/hips-image-services/hips2fits`)
returns FITS image cutouts from any HiPS survey (PanSTARRS DR1, DECaLS DR9, DSS2, 2MASS…)
given a sky position, image size, and pixel scale.

**Package used**: `astroquery.hips2fits` (part of `astroquery`, maintained by CDS)
with a fallback to the `requests` HTTP interface to the same web service.


- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **creation date**: 2026-06-23
- **last update date**: 2026-06-24 : Choose the good HIPS maps
- input : `data_SIMBAD_02/summary_visit_counts_per_star.parquet`
  (or `data_SIMBAD_01/master_stable_stars_V17-20_r1.5deg.csv`)
- CDS HiPS2FITS service : https://alasky.cds.unistra.fr/hips-image-services/hips2fits

## 1. Imports & configuration

In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm, PowerNorm

import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization import ZScaleInterval, ImageNormalize, AsinhStretch
import astroquery

# ── CDS HiPS2FITS via astroquery ─────────────────────────────────────────────
try:
    from astroquery.hips2fits import hips2fits

    HAS_ASTROQUERY_HIPS2FITS = True
    print("astroquery.hips2fits available")
except ImportError:
    HAS_ASTROQUERY_HIPS2FITS = False
    print("astroquery.hips2fits NOT found — will use requests HTTP fallback")

import requests

warnings.filterwarnings("ignore")

print(f"numpy   version : {np.__version__}")
print(f"pandas  version : {pd.__version__}")

In [ ]:
# try:
#    import ipympl  # noqa: F401

#    %matplotlib widget
#    print("ipympl found → interactive backend")
# except ImportError:
#    %matplotlib inline
#    print("ipympl NOT found → inline")

In [ ]:
%matplotlib inline

In [ ]:
VALID_HIPS_OLD = [
    "CDS/P/DSS2/color",
    "CDS/P/DSS2/red",
    "CDS/P/DSS2/blue",
    "CDS/P/PanSTARRS/DR1/r",
    "CDS/P/PanSTARRS/DR1/g",
    "CDS/P/PanSTARRS/DR1/i",
    "CDS/P/PanSTARRS/DR1/z",
    "CDS/P/PanSTARRS/DR1/y",
    "CDS/P/DES-DR2/ColorIRG",
    "CDS/P/DES-DR2/r",
    "CDS/P/DES-DR2/i",
    "CDS/P/DES-DR2/g",
]

VALID_HIPS = [
    "CDS/P/PanSTARRS/DR1/g",
    "CDS/P/PanSTARRS/DR1/r",
    "CDS/P/PanSTARRS/DR1/i",
    "CDS/P/PanSTARRS/DR1/z",
    "CDS/P/PanSTARRS/DR1/y",
]

# CANDIDATE_HIPS = ["CDS/P/PanSTARRS/DR1/r", "CDS/P/DES-DR2/r", "CDS/P/DSS2/color"]

In [ ]:
# ── Output directories ──────────────────────────────────────────────────────────────
NB_TAG = "SIMBAD_03"
DIR_DATA_IN1 = "data_SIMBAD_01"
DIR_DATA_IN2 = "data_SIMBAD_02"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"

DIR_STARS = os.path.join(DIR_DATA, "per_star")  # one file per star
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
# os.makedirs(DIR_STARS, exist_ok=True)
print(f"Data root : {os.path.abspath(DIR_DATA)}")
# print(f"Per-star  : {os.path.abspath(DIR_STARS)}")
print(f"Figs      : {os.path.abspath(DIR_FIGS)}")


# ── Cutout parameters ─────────────────────────────────────────────────────────
# Field of view for the cutout image
CUTOUT_FOV_ARCSEC = 60.0  # [adjustable] field of view in arcsec
CUTOUT_FOV_DEG = CUTOUT_FOV_ARCSEC / 3600.0
CUTOUT_NPIX = 512  # [adjustable] image size in pixels (square)


# ── Search radius (degrees) ───────────────────────────────────────────────────
# 0.5 deg ≈ the DDF half-width; increase to 1.0 deg to cover the full DDF footprint
SEARCH_RADIUS_DEG = 2.0  # [adjustable]

# ── Magnitude selection window (V band) ──────────────────────────────────────
# Rubin r-band saturation limit is ~ 16 mag; faint limit depends on purpose.
MAG_MIN = 17.0  # [adjustable]
MAG_MAX = 21.0  # [adjustable]

tag = f"V{MAG_MIN:.0f}-{MAG_MAX:.0f}_r{SEARCH_RADIUS_DEG:.1f}deg"

# ── Input catalogues ─────────────────────────────────────────────────────────
# Priority: summary table from NB02 (has visit counts); fallback to NB01 master
PATH_SUMMARY = f"{DIR_DATA_IN2}/summary_visit_counts_per_star_{tag}.parquet"
PATH_MASTER_CSV = f"{DIR_DATA_IN1}/master_stable_stars_{tag}.csv"


# ── HiPS surveys to query (CDS/Aladin) ───────────────────────────────────────
# Each entry: (hips_id, display_label, colormap)
# HiPS IDs from: https://aladin.cds.unistra.fr/hips/list
HIPS_SURVEYS = [
    ("CDS/P/PanSTARRS/DR1/g", "PanSTARRS DR1 g", "grey"),
    ("CDS/P/PanSTARRS/DR1/r", "PanSTARRS DR1 r", "grey"),
    ("CDS/P/PanSTARRS/DR1/i", "PanSTARRS DR1 i", "hot"),
    ("CDS/P/PanSTARRS/DR1/z", "PanSTARRS DR1 z", "hot"),
    ("CDS/P/PanSTARRS/DR1/y", "PanSTARRS DR1 y", "inferno"),
    # ('CDS/P/DECaLS/DR9/r',      'DECaLS DR9 r',  'gray_r'),
    ("CDS/P/DSS2/color", "DSS2 color", None),
    ("CDS/P/2MASS/J", "2MASS J", "inferno"),
]  # [adjustable]

# ── Selection: how many stars to show per DDF ─────────────────────────────────
N_STARS_PER_DDF = 16  # [adjustable] top-N by visit count per DDF
MIN_VISITS = 1  # [adjustable] skip stars with fewer total visits

# ── Output directories ─────────────────────────────────────────────────────────
NB_TAG = "SIMBAD_03"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
DIR_FITS = os.path.join(DIR_DATA, "fits_cutouts")
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
os.makedirs(DIR_FITS, exist_ok=True)
print(f"Data : {os.path.abspath(DIR_DATA)}")
print(f"FITS : {os.path.abspath(DIR_FITS)}")
print(f"Figs : {os.path.abspath(DIR_FIGS)}")

# ── Matplotlib style ──────────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": False,
        "axes.spines.top": True,
        "axes.spines.right": True,
        "font.size": 8,
    }
)


def savefig(name: str) -> None:
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")
print(f"Cutout FOV : {CUTOUT_FOV_ARCSEC} arcsec  ({CUTOUT_NPIX}x{CUTOUT_NPIX} pix)")

## 2. CDS HiPS2FITS service — overview

The **HiPS2FITS** web service, hosted at the Centre de Données astronomiques
de Strasbourg (CDS), allows retrieval of FITS image cutouts from any HiPS
(Hierarchical Progressive Survey) sky coverage.

**Service URL**: `https://alasky.cds.unistra.fr/hips-image-services/hips2fits`

**Key parameters**:

| Parameter | Description | Example |
|-----------|-------------|--------|
| `hips`    | HiPS survey identifier (from Aladin list) | `CDS/P/PanSTARRS/DR1/r` |
| `ra`      | Right Ascension of cutout centre (deg) | `150.1191` |
| `dec`     | Declination of cutout centre (deg) | `2.2058` |
| `fov`     | Field of view (deg) | `0.002778` (= 10 arcsec) |
| `width`   | Image width in pixels | `128` |
| `height`  | Image height in pixels | `128` |
| `format`  | Output format: `fits` or `jpg` or `png` | `fits` |
| `projection` | WCS projection | `TAN` |

**astroquery interface**: `astroquery.hips2fits.hips2fits.query()` wraps the HTTP call
and returns an `astropy.io.fits.HDUList`.


## 3. Helper functions for HiPS2FITS retrieval

In [ ]:
# ── CDS HiPS2FITS base URL ────────────────────────────────────────────────────
HIPS2FITS_URL = "https://alasky.cds.unistra.fr/hips-image-services/hips2fits"


def fetch_cutout_astroquery(
    hips_id: str,
    ra_deg: float,
    dec_deg: float,
    fov_deg: float = CUTOUT_FOV_DEG,
    npix: int = CUTOUT_NPIX,
) -> fits.HDUList | None:
    """
    Retrieve a FITS cutout using astroquery.hips2fits.
    Returns an HDUList or None on failure.
    """
    try:
        coord = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg, frame="icrs")
        result = hips2fits.query(
            hips=hips_id,
            ra=coord.ra,
            dec=coord.dec,
            fov=fov_deg * u.deg,
            width=npix,
            height=npix,
            projection="TAN",
            get_query_payload=False,
            format="fits",
        )
        return result
    except Exception as exc:
        print(f"    [astroquery] {hips_id}: {exc}")
        return None


def fetch_cutout_http(
    hips_id: str,
    ra_deg: float,
    dec_deg: float,
    fov_deg: float = CUTOUT_FOV_DEG,
    npix: int = CUTOUT_NPIX,
    timeout: int = 30,
) -> fits.HDUList | None:
    """
    Retrieve a FITS cutout from the CDS HiPS2FITS web service using a direct
    HTTP GET request (fallback when astroquery.hips2fits is not available).
    Returns an astropy HDUList or None on failure.
    """
    import io

    params = {
        "hips": hips_id,
        "ra": ra_deg,
        "dec": dec_deg,
        "fov": fov_deg,
        "width": npix,
        "height": npix,
        "projection": "TAN",
        "format": "fits",
        "coordsys": "icrs",
    }
    try:
        resp = requests.get(HIPS2FITS_URL, params=params, timeout=timeout)
        resp.raise_for_status()
        hdul = fits.open(io.BytesIO(resp.content))
        return hdul
    except Exception as exc:
        print(f"    [HTTP] {hips_id}: {exc}")
        return None


def fetch_cutout(
    hips_id: str,
    ra_deg: float,
    dec_deg: float,
    fov_deg: float = CUTOUT_FOV_DEG,
    npix: int = CUTOUT_NPIX,
) -> fits.HDUList | None:
    """
    Dispatch to astroquery.hips2fits if available, else HTTP fallback.
    Returns an HDUList or None.
    """
    if HAS_ASTROQUERY_HIPS2FITS:
        hdul = fetch_cutout_astroquery(hips_id, ra_deg, dec_deg, fov_deg, npix)
    else:
        hdul = fetch_cutout_http(hips_id, ra_deg, dec_deg, fov_deg, npix)
    return hdul


print("fetch_cutout() defined (astroquery or HTTP fallback).")

In [ ]:
def get_image_data(hdul: fits.HDUList) -> tuple[np.ndarray, fits.Header]:
    """
    Extract 2-D image array and FITS header from an HDUList.
    Handles both standard (PRIMARY extension) and cube (NAXIS=3) cases.
    """
    hdu = hdul[0]
    data = hdu.data
    header = hdu.header
    if data is None and len(hdul) > 1:
        hdu = hdul[1]
        data = hdu.data
        header = hdu.header
    if data is not None and data.ndim == 3:
        # Cube: take the first frame (e.g. color JPEG converted to FITS has 3 channels)
        data = data[0]
    return data, header


def make_norm(data: np.ndarray, stretch: str = "asinh") -> ImageNormalize:
    """
    Build an astropy ImageNormalize with ZScale interval and
    the requested stretch ('asinh', 'linear', 'log', 'sqrt').
    """
    interval = ZScaleInterval(contrast=0.5)
    stretch_obj = AsinhStretch(a=0.1)  # default

    if stretch == "linear":
        from astropy.visualization import LinearStretch

        stretch_obj = LinearStretch()
    elif stretch == "log":
        from astropy.visualization import LogStretch

        stretch_obj = LogStretch()
    elif stretch == "sqrt":
        from astropy.visualization import SqrtStretch

        stretch_obj = SqrtStretch()
    return ImageNormalize(data, interval=interval, stretch=stretch_obj, clip=True)


print("get_image_data() and make_norm() defined.")

## 4. Load the stable star catalogue

We load the summary table from notebook `02` (which contains per-star visit
counts and coordinates).  If it is unavailable, we fall back to the Simbad
master catalogue from notebook `01`.


In [ ]:
if os.path.exists(PATH_SUMMARY):
    df_stars = pd.read_parquet(PATH_SUMMARY)
    print(f"Loaded summary table from NB02 : {df_stars.shape}")
    # Ensure ra/dec column names match NB01 convention
    if "ra_deg" not in df_stars.columns and "ra" in df_stars.columns:
        df_stars.rename(columns={"ra": "ra_deg", "dec": "dec_deg"}, inplace=True)
elif os.path.exists(PATH_MASTER_CSV):
    df_stars = pd.read_csv(PATH_MASTER_CSV)
    print(f"Loaded master catalogue from NB01 : {df_stars.shape}")
    # Add a dummy n_visits_total if absent
    if "n_visits_total" not in df_stars.columns:
        df_stars["n_visits_total"] = 0
else:
    raise FileNotFoundError(
        f"Neither {PATH_SUMMARY} nor {PATH_MASTER_CSV} found.\n" "Run notebooks 01 and 02 first."
    )

print(f"Columns : {list(df_stars.columns)}")
df_stars.head(5)

In [ ]:
# ── Select stars with known spectral type and minimum visit count ──────────────
def has_known_sptype(sp) -> bool:
    if pd.isna(sp):
        return False
    return str(sp).strip() not in ("", "?", "unknown", "--", "nan", "None")


mask = df_stars["spectral_type"].apply(has_known_sptype) & (df_stars["n_visits_total"] >= MIN_VISITS)
df_sel = df_stars[mask].copy().reset_index(drop=True)
print(f"Stars with known SpT and >= {MIN_VISITS} visits : {len(df_sel)} / {len(df_stars)}")

# Per-DDF: keep top-N by visit count
if "field" in df_sel.columns:
    df_targets = (
        df_sel.sort_values("n_visits_total", ascending=False)
        .groupby("field")
        .head(N_STARS_PER_DDF)
        .reset_index(drop=True)
    )
else:
    df_targets = df_sel.head(N_STARS_PER_DDF * 4).reset_index(drop=True)

print(f"Target stars for cutouts : {len(df_targets)}")
cols_show = ["simbad_id", "spectral_type", "V_mag", "ra_deg", "dec_deg", "field", "n_visits_total"]
cols_show = [c for c in cols_show if c in df_targets.columns]
df_targets[cols_show]

## 5. Connectivity test — CDS HiPS2FITS service

Before the main loop, we verify that the CDS HiPS2FITS service is reachable
by downloading a single test cutout for the COSMOS field centre.


In [ ]:
TEST_RA = 150.1191  # COSMOS centre
TEST_DEC = 2.2058
TEST_HIPS = HIPS_SURVEYS[0][0]  # PanSTARRS DR1 r
# TEST_HIPS = HIPS_SURVEYS[2][0]   # DECaLS DR9 r
# TEST_HIPS = HIPS_SURVEYS[3][0]    # DSS2 color

print(f"Testing HiPS2FITS with survey : {TEST_HIPS}")
print(f"Position                      : RA={TEST_RA}, Dec={TEST_DEC}")
t0 = time.time()
hdul_test = fetch_cutout(TEST_HIPS, TEST_RA, TEST_DEC)
dt = time.time() - t0

if hdul_test is not None:
    data_test, hdr_test = get_image_data(hdul_test)
    print(f"  OK  — shape: {data_test.shape}  dtype: {data_test.dtype}  ({dt:.1f}s)")
    print(f'  CRPIX : ({hdr_test.get("CRPIX1","?")} , {hdr_test.get("CRPIX2","?")})')
    print(f'  CDELT : ({hdr_test.get("CDELT1","?")} , {hdr_test.get("CDELT2","?")})')
    print("pixel scale (arcsec):", abs(hdr_test["CDELT1"]) * 3600)

    # Quick display
    norm = make_norm(data_test)

    wcs_test = WCS(hdr_test)
    if wcs_test.pixel_n_dim > 2:
        wcs_plot = wcs_test.celestial
    else:
        wcs_plot = wcs_test

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": wcs_plot})

    if data_test.ndim == 3:
        ax.imshow(data_test, origin="lower")
    else:
        norm = make_norm(data_test)
        ax.imshow(data_test, norm=norm, cmap="gray", origin="lower")

    ax.set_xlabel("RA")
    ax.set_ylabel("Dec")

    # ⚠️ IMPORTANT : cohérence WCS utilisée ici
    x, y = wcs_plot.all_world2pix([[TEST_RA, TEST_DEC]], 0)[0]
    ax.plot(x, y, "+", color="red", ms=14, mew=2)

    # ax.set_xlabel('RA')
    # ax.set_ylabel('Dec')
    ax.set_title(f'Test cutout — COSMOS centre\n{TEST_HIPS}\n{CUTOUT_FOV_ARCSEC}" FOV')
    # ax.plot(*wcs_plot.all_world2pix([[TEST_RA, TEST_DEC]], 0)[0],
    #        '+', color='red', ms=14, mew=2, transform=ax.get_transform('world'),
    #        label='field centre')
    ax.legend(fontsize=7)
    plt.tight_layout()
    savefig("test_cutout_cosmos_centre")
    plt.show()
else:
    print("  FAILED — check your internet connection or the HiPS survey ID.")

## 6. Download and display cutouts for all target stars

For each target star we:
1. Download a FITS cutout from each HiPS survey defined in `HIPS_SURVEYS`.
2. Save the FITS file to `data_SIMBAD_03/fits_cutouts/<star_id>_<survey>.fits`.
3. Display all surveys in a single multi-panel figure, one column per survey.
4. Mark the star's exact position with a red cross using the WCS.
5. Save the figure to `figs_SIMBAD_03/`.

A **configurable stretch** (asinh, log, linear, sqrt) is applied via
`astropy.visualization.ImageNormalize` with the ZScale interval.


In [ ]:
# ── Stretch to apply to each survey panel ─────────────────────────────────────
DISPLAY_STRETCH = "asinh"  # [adjustable] 'asinh' | 'log' | 'linear' | 'sqrt'

# ── Cross-hair marker size ─────────────────────────────────────────────────────
MARKER_ARCSEC = 1.0  # [adjustable] size of the red cross in arcsec

print(f"Image stretch : {DISPLAY_STRETCH}")
print(f"Surveys       : {[s[0] for s in HIPS_SURVEYS]}")

In [ ]:
def show_cutouts_one_star(
    star_row: pd.Series,
    surveys: list,
    fov_deg: float = CUTOUT_FOV_DEG,
    npix: int = CUTOUT_NPIX,
    stretch: str = DISPLAY_STRETCH,
    save: bool = True,
) -> dict:
    """
    Download FITS cutouts for one star from all surveys and display them
    side-by-side with WCS axes and a centred red cross marker.

    Parameters
    ----------
    star_row   : pandas Series with simbad_id, ra_deg, dec_deg, spectral_type, V_mag, field
    surveys    : list of (hips_id, label, cmap) tuples
    fov_deg    : cutout field of view in degrees
    npix       : image size in pixels
    stretch    : display stretch
    save       : if True, save the figure

    Returns
    -------
    dict : {hips_id -> fits.HDUList | None}
    """
    simbad_id = star_row["simbad_id"]
    ra_deg = float(star_row["ra_deg"])
    dec_deg = float(star_row["dec_deg"])
    spectral_type = star_row.get("spectral_type", "?")
    v_mag = float(star_row.get("V_mag", np.nan))
    field = star_row.get("field", "")
    n_visits = int(star_row.get("n_visits_total", 0))

    safe_id = simbad_id.replace(" ", "_").replace("/", "-").replace(":", "-")

    n_surv = len(surveys)
    fig = plt.figure(figsize=(4 * n_surv, 4.5))
    gs = gridspec.GridSpec(1, n_surv, figure=fig, wspace=0.05)

    hduls = {}

    for col_idx, (hips_id, label, cmap) in enumerate(surveys):
        # ── Download (cache to disk) ──────────────────────────────────────────
        fits_path = os.path.join(
            DIR_FITS,
            f"{safe_id}_{hips_id.replace('/', '_')}.fits",
        )
        if os.path.exists(fits_path):
            hdul = fits.open(fits_path)
        else:
            hdul = fetch_cutout(hips_id, ra_deg, dec_deg, fov_deg, npix)
            if hdul is not None:
                hdul.writeto(fits_path, overwrite=True)
        hduls[hips_id] = hdul

        if hdul is None:
            ax = fig.add_subplot(gs[0, col_idx])
            ax.text(
                0.5,
                0.5,
                f"{label}\nNo data",
                ha="center",
                va="center",
                transform=ax.transAxes,
                fontsize=10,
                color="red",
            )
            ax.set_xticks([])
            ax.set_yticks([])
            continue

        data, header = get_image_data(hdul)

        if data is None:
            ax = fig.add_subplot(gs[0, col_idx])
            ax.text(
                0.5,
                0.5,
                f"{label}\nEmpty",
                ha="center",
                va="center",
                transform=ax.transAxes,
                fontsize=8,
                color="red",
            )
            ax.set_xticks([])
            ax.set_yticks([])
            continue

        # ── WCS axes ──────────────────────────────────────────────────────────
        wcs = WCS(header)

        if wcs.pixel_n_dim > 2:
            wcs_plot = wcs.celestial
        else:
            wcs_plot = wcs

        ax = fig.add_subplot(gs[0, col_idx], projection=wcs_plot)

        # ── Normalise & display ───────────────────────────────────────────────
        finite = data[np.isfinite(data)]
        if finite.size == 0:
            ax.text(
                0.5,
                0.5,
                f"{label}\nAll NaN",
                ha="center",
                va="center",
                transform=ax.transAxes,
                fontsize=10,
                color="red",
            )
            ax.set_xticks([])
            ax.set_yticks([])
            continue

        # norm = make_norm(data, stretch=stretch)
        norm = make_norm(data)

        if cmap is None:
            # No CMAP for "CDS/P/DSS2/color"
            ax.imshow(data, norm=norm, origin="lower")
        else:
            ax.imshow(data, norm=norm, cmap=cmap, origin="lower")

        # ── Star marker at exact sky position ─────────────────────────────────
        ax.plot(
            ra_deg,
            dec_deg,
            "+",
            color="red",
            ms=12,
            mew=1.5,
            transform=ax.get_transform("world"),
            label="target",
        )

        # ── Axis labels (only leftmost panel gets full labels) ────────────────
        ax.set_title(label, fontsize=12, pad=2)
        if col_idx == 0:
            ax.set_xlabel("RA", fontsize=7)
            ax.set_ylabel("Dec", fontsize=7)
            ax.coords[0].set_ticklabel(size=10)
            ax.coords[1].set_ticklabel(size=10)
        else:
            ax.coords[0].set_ticklabel_visible(False)
            ax.coords[1].set_ticklabel_visible(False)
            ax.coords[0].set_axislabel("")
            ax.coords[1].set_axislabel("")

    # ── Supertitle ────────────────────────────────────────────────────────────
    fig.suptitle(
        f"{simbad_id}   [{spectral_type}]   V={v_mag:.2f}   field: {field}\n"
        f'RA={ra_deg:.4f}  Dec={dec_deg:.4f}   FOV={CUTOUT_FOV_ARCSEC}"  '
        f"Rubin visits: {n_visits}",
        fontsize=12,
        y=1.01,
    )

    if save:
        savefig(f"cutout_{safe_id}")
    plt.show()
    # plt.close(fig)

    return hduls


print("show_cutouts_one_star() defined.")

In [ ]:
# ── Main loop: one figure per target star ─────────────────────────────────────
n_total = len(df_targets)
n_ok = 0
n_failed = 0

print(f"Processing {n_total} target stars...")
print("=" * 70)

for idx, star_row in df_targets.iterrows():
    print(
        f'[{idx+1:3d}/{n_total}] {star_row["simbad_id"]:40s}  '
        f'SpT={star_row.get("spectral_type","?"):6s}  '
        f'V={star_row.get("V_mag", np.nan):.2f}  '
        f'field={star_row.get("field","?")}  '
        f'N_vis={int(star_row.get("n_visits_total",0))}'
    )
    hduls = show_cutouts_one_star(star_row, HIPS_SURVEYS, save=True)

    n_retrieved = sum(1 for h in hduls.values() if h is not None)
    if n_retrieved > 0:
        n_ok += 1
    else:
        n_failed += 1

    time.sleep(0.3)  # polite delay to avoid hammering the CDS service

print("=" * 70)
print(f"Done — {n_ok} stars with at least one cutout, {n_failed} fully failed.")

## 7. Per-DDF mosaic — PanSTARRS r-band overview

For each DDF we display a mosaic of all selected stars in the PanSTARRS DR1 r-band
cutout, one thumbnail per star, arranged in a grid.
This gives a quick visual overview of the stellar environments in each field.


In [ ]:
index = 2
MOSAIC_HIPS = HIPS_SURVEYS[index][0]  # PanSTARRS DR1 r — [adjustable]
MOSAIC_HIPS_NAME = HIPS_SURVEYS[index][1]
MOSAIC_CMAP = HIPS_SURVEYS[index][2]


def make_ddf_mosaic(
    df_field: pd.DataFrame,
    field_name: str,
    hips_id: str = MOSAIC_HIPS,
    cmap: str = MOSAIC_CMAP,
    fov_deg: float = CUTOUT_FOV_DEG,
    npix: int = CUTOUT_NPIX,
) -> None:
    """
    Build and display a mosaic of cutout thumbnails for all stars in one DDF.
    """
    n_stars = len(df_field)
    if n_stars == 0:
        print(f"[{field_name}] No stars — skipping mosaic.")
        return

    ncols = min(5, n_stars)
    nrows = int(np.ceil(n_stars / ncols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(2.8 * ncols, 3.0 * nrows),
        squeeze=False,
    )

    survey_label = hips_id.split("/")[-1]

    for ax_idx, (_, star) in enumerate(df_field.iterrows()):
        row_i, col_i = divmod(ax_idx, ncols)
        ax = axes[row_i][col_i]

        safe_id = star["simbad_id"].replace(" ", "_").replace("/", "-").replace(":", "-")
        fits_path = os.path.join(
            DIR_FITS,
            f"{safe_id}_{hips_id.replace('/', '_')}.fits",
        )

        if os.path.exists(fits_path):
            hdul = fits.open(fits_path)
        else:
            hdul = fetch_cutout(hips_id, float(star["ra_deg"]), float(star["dec_deg"]), fov_deg, npix)
            if hdul is not None:
                hdul.writeto(fits_path, overwrite=True)

        if hdul is None:
            ax.text(
                0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes, fontsize=7, color="red"
            )
            ax.set_xticks([])
            ax.set_yticks([])
        else:
            data, header = get_image_data(hdul)
            finite = data[np.isfinite(data)] if data is not None else np.array([])
            if finite.size == 0:
                ax.text(
                    0.5,
                    0.5,
                    "All NaN",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                    fontsize=7,
                    color="orange",
                )
                ax.set_xticks([])
                ax.set_yticks([])
            else:
                norm = make_norm(data)
                ax.imshow(data, norm=norm, cmap=cmap, origin="lower")
                cx, cy = npix / 2, npix / 2
                ax.plot(cx, cy, "+", color="red", ms=10, mew=1.5)
                ax.set_xticks([])
                ax.set_yticks([])

        ax.set_title(
            f"{star['simbad_id']}\n" f"[{star.get('spectral_type','?')}]  V={star.get('V_mag',np.nan):.2f}",
            fontsize=6,
            pad=2,
        )

    for ax_idx in range(n_stars, nrows * ncols):
        row_i, col_i = divmod(ax_idx, ncols)
        axes[row_i][col_i].set_visible(False)

    fig.suptitle(
        f'DDF: {field_name} — {survey_label} cutouts  ({CUTOUT_FOV_ARCSEC}" FOV)\n'
        f"{n_stars} stable stars with known spectral type",
        fontsize=9,
        y=1.01,
    )
    plt.tight_layout()
    savefig(f"mosaic_{field_name.replace('-', '_')}_{survey_label}")
    plt.show()
    # plt.close(fig)


# ── Run mosaics for each DDF field ────────────────────────────────────────────
if "field" in df_targets.columns:
    for field_name in sorted(df_targets["field"].unique()):
        df_field = df_targets[df_targets["field"] == field_name]
        print(f"\nMosaic for {field_name} ({len(df_field)} stars)...")
        make_ddf_mosaic(df_field, field_name)
else:
    print("No field column — making a single mosaic.")
    make_ddf_mosaic(df_targets, "all_fields")

## 8. Interactive single-star viewer

Use this cell to inspect any star by its Simbad ID or row index.
Adjust `STAR_IDX` or `STAR_ID` and re-run to display a fresh set of cutouts.


In [ ]:
# ── Pick a star to inspect ────────────────────────────────────────────────────
STAR_IDX = 1  # [adjustable] row index in df_targets
STAR_ID = None  # [adjustable] e.g. '2MASS J10005...' — overrides STAR_IDX if set

# ── Select the star ───────────────────────────────────────────────────────────
if STAR_ID is not None and "simbad_id" in df_targets.columns:
    matches = df_targets[df_targets["simbad_id"] == STAR_ID]
    if matches.empty:
        print(f'STAR_ID "{STAR_ID}" not found in df_targets — using STAR_IDX={STAR_IDX}')
        star_inspect = df_targets.iloc[STAR_IDX]
    else:
        star_inspect = matches.iloc[0]
else:
    star_inspect = df_targets.iloc[STAR_IDX]

print("Inspecting star:")
for col in ["simbad_id", "spectral_type", "V_mag", "ra_deg", "dec_deg", "field", "n_visits_total"]:
    if col in star_inspect.index:
        print(f"  {col:18s}: {star_inspect[col]}")

# ── Fetch and display all surveys for this star ───────────────────────────────
_ = show_cutouts_one_star(star_inspect, HIPS_SURVEYS, save=True)

## 9. FITS header inspection

Inspect the WCS header of a downloaded FITS cutout.
This documents the pixel scale and coordinate system returned by HiPS2FITS.


In [ ]:
# Load the first available FITS file from DIR_FITS and print the header
fits_files = sorted([os.path.join(DIR_FITS, f) for f in os.listdir(DIR_FITS) if f.endswith(".fits")])

if not fits_files:
    print("No FITS files found in", DIR_FITS)
else:
    example_fits = fits_files[0]
    print(f"Example FITS: {example_fits}")
    with fits.open(example_fits) as hdul:
        hdul.info()
        hdr = hdul[0].header
        print(repr(hdr))

    # Compute the pixel scale
    cdelt1 = abs(hdr.get("CDELT1", np.nan))  # deg/pix
    cdelt2 = abs(hdr.get("CDELT2", np.nan))
    print(f"\nPixel scale : {cdelt1 * 3600:.4f} arcsec/pix (RA)  x  {cdelt2 * 3600:.4f} arcsec/pix (Dec)")
    print(f'Image size  : {hdr.get("NAXIS1","?")} x {hdr.get("NAXIS2","?")} pixels')
    print(
        f'FOV         : {hdr.get("NAXIS1",1) * cdelt1 * 3600:.1f} x '
        f'{hdr.get("NAXIS2",1) * cdelt2 * 3600:.1f} arcsec'
    )

## 10. Summary

| Item | Value |
|------|-------|
| CDS HiPS2FITS service | `https://alasky.cds.unistra.fr/hips-image-services/hips2fits` |
| astroquery interface  | `astroquery.hips2fits.hips2fits.query()` |
| Surveys queried       | PanSTARRS DR1 r/i, DECaLS DR9 r, DSS2 color, 2MASS J |
| Cutout FOV            | configurable via `CUTOUT_FOV_ARCSEC` (default 10 arcsec) |
| Image size            | configurable via `CUTOUT_NPIX` (default 128 x 128 pix) |
| FITS files saved to   | `data_SIMBAD_03/fits_cutouts/` |
| Figures saved to      | `figs_SIMBAD_03/` |

### Next steps

- **`04_crossmatch_rubin_cutouts.ipynb`** — download Rubin Butler DIA stamp
  cutouts for the same stars and compare with the external survey images.
- **`05_psf_morphology.ipynb`** — measure PSF size and shape on the CDS
  cutouts to verify that the star is isolated and unresolved.
